# 04 — Question Generation Demo

Full pipeline: **chunks (MongoDB) → select → prompt → Gemini LLM → parse → questions**

This notebook demonstrates Week 4:
1. Generate questions for a single Java chapter
2. Generate for the entire Java textbook
3. Generate for the C textbook
4. Save all questions + manual evaluation template

In [1]:
import json
import logging
import os
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")

from hsc_edu.storage.mongo_store import MongoChunkStore

mongo = MongoChunkStore()
print(f"MongoDB chunks: {mongo.count()}")
print(f"Subjects: {mongo.distinct_values('subject')}")
print(f"GOOGLE_API_KEY set: {bool(os.environ.get('GOOGLE_API_KEY'))}")

MongoDB chunks: 437
Subjects: ['Lập trình C', 'Lập trình Java']
GOOGLE_API_KEY set: True


## 1. Single chapter test — Java "7.7. ĐA HÌNH"

In [2]:
from hsc_edu.generation import generate_questions, GeneratedQuestion

qs_poly = generate_questions(
    subject="Lập trình Java",
    chapter="7.7. ĐA HÌNH",
    num_questions=3,
)

print(f"Generated {len(qs_poly)} questions:\n")
for i, q in enumerate(qs_poly, 1):
    print(f"[{i}] ({q.difficulty}) {q.question}")
    print(f"    Answer: {q.suggested_answer[:150]}..." if len(q.suggested_answer) > 150 else f"    Answer: {q.suggested_answer}")
    print(f"    Source: {q.source}  |  Bloom: {q.bloom_level}")
    print(f"    Keywords: {q.keywords}")
    print()

INFO | hsc_edu.generation.question_generator | Selected 3 chunks for generation (subject='Lập trình Java', chapter='7.7. ĐA HÌNH')
INFO | google_genai.models | AFC is enabled with max remote calls: 10.
INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-latest:generateContent "HTTP/1.1 200 OK"
INFO | hsc_edu.generation.llm_client | LLM generated 1869 chars (model=gemini-flash-latest, attempt=1)
INFO | hsc_edu.generation.question_generator | Generated 3 questions for subject='Lập trình Java' chapter='7.7. ĐA HÌNH'


Generated 3 questions:

[1] (Nhận biết) Theo giáo trình, đa hình trong lập trình hướng đối tượng có hai đặc điểm chính nào?
    Answer: Đa hình có hai đặc điểm: (1) Các đối tượng thuộc các lớp dẫn xuất khác nhau có thể được đối xử như nhau như thể chúng là đối tượng thuộc lớp cơ sở; (2...
    Source: Mục 7.7, trang 112-113  |  Bloom: Remember
    Keywords: ['đặc điểm đa hình', 'lớp dẫn xuất', 'lớp cơ sở', 'thông điệp']

[2] (Thông hiểu) Trong ví dụ về mảng đa hình 'Animal[] animals', điều gì xảy ra khi ta duyệt mảng và gọi phương thức 'eat()' trên từng phần tử?
    Answer: Khi duyệt mảng, mặc dù ta gọi phương thức từ tham chiếu kiểu Animal, nhưng mỗi đối tượng (như Dog, Cat, Wolf...) sẽ thực hiện đúng phiên bản phương th...
    Source: Mục 7.7, trang 111  |  Bloom: Understand
    Keywords: ['mảng đa hình', 'tham chiếu', 'đối tượng', 'phương thức']

[3] (Vận dụng) Tại sao việc sử dụng tham số đa hình trong phương thức 'giveShot(Animal a)' của lớp 'Vet' lại giúp chương trình dễ bảo trì v

## 2. Limited textbook (API-safe) — Java.pdf

In [8]:
import time

from hsc_edu.generation import generate_questions

JAVA_QUESTIONS_PER_CHAPTER = 1
JAVA_MAX_CHAPTERS = 3  # tăng để sinh nhiều hơn, giảm để tiết kiệm API

all_chunks_java = mongo.get_chunks_by_filter(subject="Lập trình Java")
chapters_java = sorted({c.chapter for c in all_chunks_java if c.chapter})
chapters_java = chapters_java[:JAVA_MAX_CHAPTERS]

qs_java = []
for i, ch in enumerate(chapters_java, 1):
    qs_java.extend(
        generate_questions(
            subject="Lập trình Java",
            chapter=ch,
            num_questions=JAVA_QUESTIONS_PER_CHAPTER,
            max_context_chunks=15,
            mongo=mongo,
        )
    )
    if i < len(chapters_java):
        time.sleep(1)

print(
    f"\nJava (limited): {len(qs_java)} questions from {len({q.chapter for q in qs_java})} chapters"
)
diff_dist = Counter(q.difficulty for q in qs_java)
print(f"Difficulty distribution: {dict(diff_dist)}")

INFO | hsc_edu.generation.question_generator | Selected 1 chunks for generation (subject='Lập trình Java', chapter='1.1. KHÁI NIỆM CƠ BẢN')
INFO | google_genai.models | AFC is enabled with max remote calls: 10.


INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-latest:generateContent "HTTP/1.1 503 Service Unavailable"
WARNING | hsc_edu.generation.llm_client | LLM call failed (attempt 1/3): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}. Retrying in 2s…
INFO | google_genai.models | AFC is enabled with max remote calls: 10.
INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-latest:generateContent "HTTP/1.1 200 OK"
INFO | hsc_edu.generation.llm_client | LLM generated 711 chars (model=gemini-flash-latest, attempt=2)
INFO | hsc_edu.generation.question_generator | Generated 1 questions for subject='Lập trình Java' chapter='1.1. KHÁI NIỆM CƠ BẢN'
INFO | hsc_edu.generation.question_generator | Selected 3 chunks for generation (subject='Lập trình Java', 


Java (limited): 10 questions from 3 chapters
Difficulty distribution: {'Thông hiểu': 6, 'Nhận biết': 1, 'Vận dụng': 2, 'Vận dụng cao': 1}


## 3. Limited textbook (API-safe) — C.pdf

In [9]:
import time

from hsc_edu.generation import generate_questions

C_QUESTIONS_PER_CHAPTER = 1
C_MAX_CHAPTERS = 3

all_chunks_c = mongo.get_chunks_by_filter(subject="Lập trình C")
chapters_c = sorted({c.chapter for c in all_chunks_c if c.chapter})
chapters_c = chapters_c[:C_MAX_CHAPTERS]

qs_c = []
for i, ch in enumerate(chapters_c, 1):
    qs_c.extend(
        generate_questions(
            subject="Lập trình C",
            chapter=ch,
            num_questions=C_QUESTIONS_PER_CHAPTER,
            max_context_chunks=15,
            mongo=mongo,
        )
    )
    if i < len(chapters_c):
        time.sleep(1)

print(
    f"\nC (limited): {len(qs_c)} questions from {len({q.chapter for q in qs_c})} chapters"
)
diff_dist_c = Counter(q.difficulty for q in qs_c)
print(f"Difficulty distribution: {dict(diff_dist_c)}")

INFO | hsc_edu.generation.question_generator | Selected 1 chunks for generation (subject='Lập trình C', chapter='1.3.2. Sử dụng lưu đồ (Flowchart)')
INFO | google_genai.models | AFC is enabled with max remote calls: 10.
INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-latest:generateContent "HTTP/1.1 200 OK"
INFO | hsc_edu.generation.llm_client | LLM generated 599 chars (model=gemini-flash-latest, attempt=1)
INFO | hsc_edu.generation.question_generator | Generated 1 questions for subject='Lập trình C' chapter='1.3.2. Sử dụng lưu đồ (Flowchart)'
INFO | hsc_edu.generation.question_generator | Selected 6 chunks for generation (subject='Lập trình C', chapter='2.1. Cài đặt và sử dụng Dev-C++')
INFO | google_genai.models | AFC is enabled with max remote calls: 10.
INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-latest:generateContent "HTTP/1.1 200 OK"
INFO | hsc_edu.generation.llm_cli


C (limited): 9 questions from 3 chapters
Difficulty distribution: {'Vận dụng': 4, 'Thông hiểu': 3, 'Nhận biết': 2}


## 4. Combine & save all questions

In [10]:
all_questions = qs_java + qs_c
print(f"Total questions: {len(all_questions)}")
print(f"  Java: {len(qs_java)}")
print(f"  C:    {len(qs_c)}")

output_path = Path("../data/questions/after_fix2.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        [q.model_dump() for q in all_questions],
        f, ensure_ascii=False, indent=2,
    )

print(f"\nSaved to {output_path}")

print("\n--- Sample questions (first 10) ---\n")
for i, q in enumerate(all_questions[:10], 1):
    print(f"[{i}] ({q.subject} | {q.chapter}) [{q.difficulty}]")
    print(f"    Q: {q.question}")
    print(f"    A: {q.suggested_answer[:120]}{'...' if len(q.suggested_answer) > 120 else ''}")
    print()

Total questions: 19
  Java: 10
  C:    9

Saved to ..\data\questions\after_fix2.json

--- Sample questions (first 10) ---

[1] (Lập trình Java | 1.1. KHÁI NIỆM CƠ BẢN) [Thông hiểu]
    Q: Dựa trên ví dụ về việc rút tiền tại máy ATM trong giáo trình, hãy giải thích cơ chế hoạt động của một chương trình hướng đối tượng trong thời gian chạy (runtime).
    A: Trong thời gian chạy, chương trình OOP là một tập hợp các đối tượng gửi thông điệp cho nhau để yêu cầu và thực hiện dịch...

[2] (Lập trình Java | 1.2. ĐỐI TƯỢNG VÀ LỚP) [Thông hiểu]
    Q: Dựa trên giáo trình, hãy giải thích mối quan hệ giữa 'lớp' (class) và 'đối tượng' (object). Tại sao lớp được coi là một khuôn mẫu?
    A: Mối quan hệ này tương tự như giữa kiểu dữ liệu và biến. Lớp là đặc tả các đặc điểm chung (thuộc tính và phương thức), đó...

[3] (Lập trình Java | 1.2. ĐỐI TƯỢNG VÀ LỚP) [Nhận biết]
    Q: Khi một đối tượng được tạo ra, hệ thống tự động thực hiện hành động gì và đối tượng đó sẽ duy trì những đặc tính riêng biệt n

## 5. Manual evaluation — 20 questions

Select 20 random questions for manual evaluation.
Fill in the **Rating** column: `Đúng` / `Chưa tốt` / `Sai`

In [11]:
import random

random.seed(42)
eval_sample = random.sample(all_questions, min(20, len(all_questions)))

print("| # | Subject | Chapter | Difficulty | Question | Answer (preview) | Rating |")
print("|---|---------|---------|------------|----------|------------------|--------|")
for i, q in enumerate(eval_sample, 1):
    subj_short = q.subject.replace("Lập trình ", "")
    ch_short = q.chapter[:30] + "..." if len(q.chapter) > 30 else q.chapter
    q_short = q.question[:60] + "..." if len(q.question) > 60 else q.question
    a_short = q.suggested_answer[:50] + "..." if len(q.suggested_answer) > 50 else q.suggested_answer
    print(f"| {i} | {subj_short} | {ch_short} | {q.difficulty} | {q_short} | {a_short} | |")

| # | Subject | Chapter | Difficulty | Question | Answer (preview) | Rating |
|---|---------|---------|------------|----------|------------------|--------|
| 1 | Java | 1.2. ĐỐI TƯỢNG VÀ LỚP | Vận dụng | Tại sao trong một thiết kế hướng đối tượng tốt, các đối tượn... | Vì đối tượng bên ngoài chỉ cần biết đối tượng đó c... | |
| 2 | Java | 1.1. KHÁI NIỆM CƠ BẢN | Thông hiểu | Dựa trên ví dụ về việc rút tiền tại máy ATM trong giáo trình... | Trong thời gian chạy, chương trình OOP là một tập ... | |
| 3 | Java | 1.3. CÁC NGUYÊN TẮC TRỤ CỘT | Vận dụng | Giải thích cơ chế 'Đa hình' (polymorphism) thông qua ví dụ v... | Đa hình là khả năng cùng một thông điệp được hiểu ... | |
| 4 | Java | 1.3. CÁC NGUYÊN TẮC TRỤ CỘT | Thông hiểu | Tại sao nói 'Thừa kế' (inheritance) là một hình thức tái sử ... | Vì thừa kế cho phép xây dựng một lớp mới bằng cách... | |
| 5 | C | 2.2. Các ví dụ làm quen với C | Thông hiểu | Khi sử dụng chỉ thị #define để định nghĩa một hằng số (ví dụ... | Trước define phải c

## 6. Statistics

In [12]:
print(f"=== Generation Statistics ===")
print(f"Total questions: {len(all_questions)}")
print(f"")

# By subject
by_subject = Counter(q.subject for q in all_questions)
print("By subject:")
for subj, cnt in by_subject.most_common():
    print(f"  {subj}: {cnt}")

# By difficulty
print("\nBy difficulty:")
by_diff = Counter(q.difficulty for q in all_questions)
for diff, cnt in by_diff.most_common():
    print(f"  {diff}: {cnt}")

# By bloom level
print("\nBy Bloom level:")
by_bloom = Counter(q.bloom_level for q in all_questions)
for bl, cnt in by_bloom.most_common():
    print(f"  {bl}: {cnt}")

# Chapter coverage
java_chapters_covered = len({q.chapter for q in qs_java if q.chapter})
c_chapters_covered = len({q.chapter for q in qs_c if q.chapter})
print(f"\nChapter coverage:")
print(f"  Java: {java_chapters_covered} chapters")
print(f"  C:    {c_chapters_covered} chapters")

=== Generation Statistics ===
Total questions: 19

By subject:
  Lập trình Java: 10
  Lập trình C: 9

By difficulty:
  Thông hiểu: 9
  Vận dụng: 6
  Nhận biết: 3
  Vận dụng cao: 1

By Bloom level:
  Understand: 9
  Apply: 6
  Remember: 3
  Analyze: 1

Chapter coverage:
  Java: 3 chapters
  C:    3 chapters
